In [ ]:
# import os
# import json
# import time
# from pathlib import Path
# from datetime import datetime

# import pandas as pd
# from openai import OpenAI

# # ============================================================
# # 0. 使用者設定區：主要改這裡即可
# # ============================================================

# MODE = "compare"
# # 可選：
# # "ai"          自動執行：抽樣 + AI 評分 + 統計分布
# # "sample"      只從主測試集隨機抽樣
# # "ai_score"    只對指定抽樣資料進行 GPT-4o-mini 評分
# # "count"       只統計指定 AI 評分檔案的分數分布
# # "compare"     AI 與人工評分比較，產生混淆矩陣與指標

# MAIN_TESTSET_CSV = "ragas_eval_with_response_2026-05-22.csv"

# RESULT_ROOT = "rag_ai_manual_eval_results"

# SAMPLE_SIZE = 10
# RANDOM_SEED = 36

# MODEL_NAME = "gpt-4o-mini"
# SLEEP_SEC = 0

# SCORE_COLUMN = "ai_score"

# MANUAL_SAMPLE_CSV = "rag_sample_10.csv"
# MANUAL_AI_SCORED_CSV = "ai_scored_10.csv"
# MANUAL_HUMAN_AI_SCORED_CSV = "rag_ai_manual_eval_results/20260522/v2/ai_scored_10.csv"


# # ============================================================
# # 1. OpenAI Client
# # ============================================================

# client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


# # ============================================================
# # 2. 評分 Prompt
# # ============================================================

# SYSTEM_PROMPT = """
# 你是一位協助碩士論文進行 RAG 系統評估的學術評分者。
# 請根據「使用者問題 user_input」、「RAG 系統回答 response」與「檢索內容 retrieved_contexts」評估回答品質。

# 請主要判斷：
# 1. RAG 系統回答是否有回應使用者問題。
# 2. RAG 系統回答的主要內容是否能由 retrieved_contexts 支持。
# 3. RAG 系統回答是否包含 retrieved_contexts 無法支持的幻覺、錯誤或過度推論。

# 評分標準如下：
# 1 分：回答與使用者問題無關，且無法由 retrieved_contexts 支持。
# 2 分：回答僅少部分回應問題，或大多數內容缺乏 retrieved_contexts 支持。
# 3 分：回答大致回應問題，但內容不完整，或部分敘述無法由 retrieved_contexts 明確支持。
# 4 分：回答能回應問題，且大部分內容可由 retrieved_contexts 支持，僅有輕微不完整或表述不精確。
# 5 分：回答完整回應問題，且主要內容皆可由 retrieved_contexts 清楚支持，無明顯幻覺或錯誤。

# 請只根據提供的 user_input、response 與 retrieved_contexts 評分，不要自行補充外部知識。
# 請輸出 JSON。
# """.strip()


# # ============================================================
# # 3. 輸出資料夾管理
# # ============================================================


# def create_run_dir(result_root: str = RESULT_ROOT) -> Path:
#     today = datetime.now().strftime("%Y%m%d")
#     date_dir = Path(result_root) / today
#     date_dir.mkdir(parents=True, exist_ok=True)

#     existing_versions = [
#         p
#         for p in date_dir.iterdir()
#         if p.is_dir() and p.name.startswith("v") and p.name[1:].isdigit()
#     ]

#     if not existing_versions:
#         next_version = 1
#     else:
#         version_numbers = [int(p.name[1:]) for p in existing_versions]
#         next_version = max(version_numbers) + 1

#     run_dir = date_dir / f"v{next_version}"
#     run_dir.mkdir(parents=True, exist_ok=False)
#     return run_dir


# def get_output_paths(run_dir: Path) -> dict:
#     return {
#         "sample_csv": run_dir / "rag_sample_10.csv",
#         "ai_scored_csv": run_dir / "ai_scored_10.csv",
#         "score_distribution_csv": run_dir / "score_distribution.csv",
#         "binary_distribution_csv": run_dir / "binary_distribution.csv",
#         "run_info_json": run_dir / "run_info.json",
#         "prompt_txt": run_dir / "prompt_used.txt",
#         "compare_csv": run_dir / "compare_result_10.csv",
#         "compare_metrics_csv": run_dir / "compare_result_10_metrics.csv",
#     }


# # ============================================================
# # 4. 分數轉二元判定
# # ============================================================


# def score_to_binary(score):
#     try:
#         score = int(score)
#     except Exception:
#         return None

#     if score in [1, 2, 3]:
#         return "未達可接受回答"
#     if score in [4, 5]:
#         return "達可接受回答"
#     return None


# # ============================================================
# # 5. 從主測試集隨機抽樣
# # ============================================================


# def sample_from_main_testset(input_csv, output_csv, sample_size=10, random_seed=42):
#     df = pd.read_csv(input_csv, encoding="utf-8-sig")

#     required_cols = [
#         "user_input",
#         "response",
#         "retrieved_contexts",
#     ]

#     missing_cols = [col for col in required_cols if col not in df.columns]
#     if missing_cols:
#         raise ValueError(f"主測試集缺少必要欄位：{missing_cols}")

#     if "status" in df.columns:
#         success_values = ["success", "completed", "ok", "SUCCESS", "Completed", "OK"]
#         success_df = df[df["status"].isin(success_values)].copy()

#         if len(success_df) >= sample_size:
#             df_for_sample = success_df
#         else:
#             print("提醒：status 成功資料不足，將改從全部資料抽樣。")
#             df_for_sample = df
#     else:
#         df_for_sample = df

#     if len(df_for_sample) < sample_size:
#         raise ValueError(
#             f"可抽樣資料只有 {len(df_for_sample)} 筆，小於 sample_size={sample_size}"
#         )

#     sample_df = df_for_sample.sample(n=sample_size, random_state=random_seed).copy()

#     sample_df.insert(0, "sample_id", [f"Q{i+1}" for i in range(len(sample_df))])
#     sample_df.insert(1, "original_index", sample_df.index)

#     front_cols = [
#         "sample_id",
#         "original_index",
#         "user_input",
#         "response",
#         "retrieved_contexts",
#     ]

#     if "retrieved_metadata" in sample_df.columns:
#         front_cols.append("retrieved_metadata")

#     if "status" in sample_df.columns:
#         front_cols.append("status")

#     other_cols = [col for col in sample_df.columns if col not in front_cols]
#     sample_df = sample_df[front_cols + other_cols]

#     sample_df.to_csv(output_csv, index=False, encoding="utf-8-sig")

#     print(f"已從 {input_csv} 隨機抽出 {sample_size} 題")
#     print(f"抽樣結果已儲存至：{output_csv}")
#     print("抽樣 sample_id：")
#     print(sample_df["sample_id"].tolist())

#     return sample_df


# # ============================================================
# # 6. 單題 AI 評分
# # ============================================================


# def evaluate_one_row(row, model="gpt-4o-mini"):
#     user_input = str(row.get("user_input", ""))
#     response = str(row.get("response", ""))
#     retrieved_contexts = str(row.get("retrieved_contexts", ""))

#     user_prompt = f"""
# 【使用者問題 user_input】
# {user_input}

# 【RAG 系統回答 response】
# {response}

# 【檢索內容 retrieved_contexts】
# {retrieved_contexts}

# 請根據「是否回應使用者問題」與「是否受到 retrieved_contexts 支持」給出 1 至 5 分評分。
# """.strip()

#     response_format = {
#         "type": "json_schema",
#         "json_schema": {
#             "name": "rag_answer_validity_score",
#             "strict": True,
#             "schema": {
#                 "type": "object",
#                 "properties": {
#                     "score": {
#                         "type": "integer",
#                         "description": "回答支持性與問題回應性評分，範圍為 1 到 5 分",
#                         "minimum": 1,
#                         "maximum": 5,
#                     },
#                     "reason": {
#                         "type": "string",
#                         "description": "簡短說明評分理由",
#                     },
#                 },
#                 "required": ["score", "reason"],
#                 "additionalProperties": False,
#             },
#         },
#     }

#     completion = client.chat.completions.create(
#         model=model,
#         messages=[
#             {"role": "system", "content": SYSTEM_PROMPT},
#             {"role": "user", "content": user_prompt},
#         ],
#         response_format=response_format,
#         temperature=0,
#     )

#     content = completion.choices[0].message.content
#     result = json.loads(content)

#     return result["score"], result["reason"]


# # ============================================================
# # 7. 批次 AI 評分
# # ============================================================


# def run_ai_scoring(input_csv, output_csv, model="gpt-4o-mini", sleep_sec=0):
#     df = pd.read_csv(input_csv, encoding="utf-8-sig")

#     required_cols = ["user_input", "response", "retrieved_contexts"]
#     missing_cols = [col for col in required_cols if col not in df.columns]
#     if missing_cols:
#         raise ValueError(f"缺少必要欄位：{missing_cols}")

#     if "sample_id" not in df.columns:
#         df.insert(0, "sample_id", [f"Q{i+1}" for i in range(len(df))])

#     ai_scores = []
#     ai_reasons = []

#     for idx, row in df.iterrows():
#         current_id = row.get("sample_id", f"Q{idx + 1}")
#         print(f"正在評分第 {idx + 1}/{len(df)} 題：{current_id}")

#         try:
#             score, reason = evaluate_one_row(row, model=model)
#         except Exception as e:
#             score = None
#             reason = f"評分失敗：{e}"

#         ai_scores.append(score)
#         ai_reasons.append(reason)

#         time.sleep(sleep_sec)

#     df["ai_score"] = ai_scores
#     df["ai_reason"] = ai_reasons
#     df["ai_binary"] = df["ai_score"].apply(score_to_binary)

#     df.to_csv(output_csv, index=False, encoding="utf-8-sig")

#     print(f"\nAI 評分完成，已儲存至：{output_csv}")
#     return df


# # ============================================================
# # 8. 分數分布統計
# # ============================================================


# def build_score_distribution_tables(df, score_column):
#     numeric_scores = pd.to_numeric(df[score_column], errors="coerce")

#     score_df = (
#         numeric_scores.value_counts()
#         .reindex([1, 2, 3, 4, 5], fill_value=0)
#         .rename_axis("score")
#         .reset_index(name="count")
#     )

#     binary_column = score_column.replace("_score", "_binary")
#     if binary_column not in df.columns:
#         df[binary_column] = df[score_column].apply(score_to_binary)

#     binary_df = (
#         df[binary_column]
#         .value_counts()
#         .reindex(["未達可接受回答", "達可接受回答"], fill_value=0)
#         .rename_axis("binary_label")
#         .reset_index(name="count")
#     )

#     return score_df, binary_df


# def print_score_distribution(
#     df, score_column, score_output_csv=None, binary_output_csv=None
# ):
#     print(f"\n{score_column} 分數分布：")

#     score_df, binary_df = build_score_distribution_tables(df, score_column)

#     print(score_df)

#     print(f"\n{score_column.replace('_score', '_binary')} 二元判定分布：")
#     print(binary_df)

#     if score_output_csv:
#         score_df.to_csv(score_output_csv, index=False, encoding="utf-8-sig")
#         print(f"\n分數分布 CSV 已輸出：{score_output_csv}")

#     if binary_output_csv:
#         binary_df.to_csv(binary_output_csv, index=False, encoding="utf-8-sig")
#         print(f"二元判定分布 CSV 已輸出：{binary_output_csv}")


# def count_scores(
#     input_csv, score_column, score_output_csv=None, binary_output_csv=None
# ):
#     df = pd.read_csv(input_csv, encoding="utf-8-sig")

#     if score_column not in df.columns:
#         raise ValueError(f"找不到欄位：{score_column}")

#     print_score_distribution(
#         df=df,
#         score_column=score_column,
#         score_output_csv=score_output_csv,
#         binary_output_csv=binary_output_csv,
#     )


# # ============================================================
# # 9. AI 與人工比較
# # ============================================================


# def compare_ai_human(input_csv, output_csv=None, metrics_output_csv=None):
#     df = pd.read_csv(input_csv, encoding="utf-8-sig")

#     required_cols = ["ai_score", "human_score"]
#     missing_cols = [col for col in required_cols if col not in df.columns]
#     if missing_cols:
#         raise ValueError(f"缺少必要欄位：{missing_cols}")

#     df["ai_binary"] = df["ai_score"].apply(score_to_binary)
#     df["human_binary"] = df["human_score"].apply(score_to_binary)

#     valid_df = df.dropna(subset=["ai_binary", "human_binary"]).copy()

#     tp = len(
#         valid_df[
#             (valid_df["human_binary"] == "達可接受回答")
#             & (valid_df["ai_binary"] == "達可接受回答")
#         ]
#     )

#     fp = len(
#         valid_df[
#             (valid_df["human_binary"] == "未達可接受回答")
#             & (valid_df["ai_binary"] == "達可接受回答")
#         ]
#     )

#     fn = len(
#         valid_df[
#             (valid_df["human_binary"] == "達可接受回答")
#             & (valid_df["ai_binary"] == "未達可接受回答")
#         ]
#     )

#     tn = len(
#         valid_df[
#             (valid_df["human_binary"] == "未達可接受回答")
#             & (valid_df["ai_binary"] == "未達可接受回答")
#         ]
#     )

#     total = tp + fp + fn + tn

#     precision = tp / (tp + fp) if (tp + fp) > 0 else 0
#     recall = tp / (tp + fn) if (tp + fn) > 0 else 0
#     f1 = (
#         2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
#     )
#     accuracy = (tp + tn) / total if total > 0 else 0

#     same_score_count = len(valid_df[valid_df["ai_score"] == valid_df["human_score"]])
#     same_binary_count = len(valid_df[valid_df["ai_binary"] == valid_df["human_binary"]])

#     exact_score_agreement = same_score_count / len(valid_df) if len(valid_df) > 0 else 0
#     binary_agreement = same_binary_count / len(valid_df) if len(valid_df) > 0 else 0

#     print("\n混淆矩陣：")
#     print("以人工評分為標準答案，AI 評分為預測結果")
#     print("正類：達可接受回答")
#     print()
#     print("                                  AI 判定")
#     print("人工判定                    達可接受回答       未達可接受回答")
#     print(f"達可接受回答       TP        {tp:<12} FN        {fn:<12}")
#     print(f"未達可接受回答     FP        {fp:<12} TN        {tn:<12}")

#     print("\n評估指標：")
#     print(f"Precision（精確率）: {precision:.4f}")
#     print(f"Recall（召回率）   : {recall:.4f}")
#     print(f"F1 分數             : {f1:.4f}")
#     print(f"Accuracy（正確率） : {accuracy:.4f}")

#     print("\n一致性補充：")
#     print(
#         f"完全同分題數：{same_score_count}/{len(valid_df)}，比例：{exact_score_agreement:.4f}"
#     )
#     print(
#         f"二元判定一致題數：{same_binary_count}/{len(valid_df)}，比例：{binary_agreement:.4f}"
#     )

#     metrics = pd.DataFrame(
#         [
#             {
#                 "TP": tp,
#                 "FP": fp,
#                 "FN": fn,
#                 "TN": tn,
#                 "Precision": precision,
#                 "Recall": recall,
#                 "F1": f1,
#                 "Accuracy": accuracy,
#                 "ExactScoreAgreement": exact_score_agreement,
#                 "BinaryAgreement": binary_agreement,
#                 "ValidN": len(valid_df),
#             }
#         ]
#     )

#     if output_csv:
#         valid_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
#         print(f"\n已輸出比較結果：{output_csv}")

#     if metrics_output_csv:
#         metrics.to_csv(metrics_output_csv, index=False, encoding="utf-8-sig")
#         print(f"已輸出指標結果：{metrics_output_csv}")

#     return metrics


# # ============================================================
# # 10. 儲存執行資訊與 Prompt
# # ============================================================


# def save_run_info(paths, run_dir):
#     run_info = {
#         "mode": MODE,
#         "main_testset_csv": MAIN_TESTSET_CSV,
#         "result_root": RESULT_ROOT,
#         "sample_size": SAMPLE_SIZE,
#         "random_seed": RANDOM_SEED,
#         "model_name": MODEL_NAME,
#         "sleep_sec": SLEEP_SEC,
#         "score_column": SCORE_COLUMN,
#         "output_dir": str(run_dir),
#         "sample_csv": str(paths["sample_csv"]),
#         "ai_scored_csv": str(paths["ai_scored_csv"]),
#         "score_distribution_csv": str(paths["score_distribution_csv"]),
#         "binary_distribution_csv": str(paths["binary_distribution_csv"]),
#         "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
#     }

#     with open(paths["run_info_json"], "w", encoding="utf-8") as f:
#         json.dump(run_info, f, ensure_ascii=False, indent=2)

#     with open(paths["prompt_txt"], "w", encoding="utf-8") as f:
#         f.write(SYSTEM_PROMPT)

#     print(f"執行資訊已輸出：{paths['run_info_json']}")
#     print(f"評分 Prompt 已輸出：{paths['prompt_txt']}")


# # ============================================================
# # 11. MODE = "ai"：自動執行抽樣 + AI 評分 + 分數統計
# # ============================================================


# def run_ai_pipeline():
#     run_dir = create_run_dir(RESULT_ROOT)
#     paths = get_output_paths(run_dir)

#     print("=" * 70)
#     print("開始執行 AI 評分流程")
#     print(f"輸出資料夾：{run_dir}")
#     print("=" * 70)

#     sample_from_main_testset(
#         input_csv=MAIN_TESTSET_CSV,
#         output_csv=paths["sample_csv"],
#         sample_size=SAMPLE_SIZE,
#         random_seed=RANDOM_SEED,
#     )

#     scored_df = run_ai_scoring(
#         input_csv=paths["sample_csv"],
#         output_csv=paths["ai_scored_csv"],
#         model=MODEL_NAME,
#         sleep_sec=SLEEP_SEC,
#     )

#     print_score_distribution(
#         df=scored_df,
#         score_column=SCORE_COLUMN,
#         score_output_csv=paths["score_distribution_csv"],
#         binary_output_csv=paths["binary_distribution_csv"],
#     )

#     save_run_info(paths, run_dir)

#     print("=" * 70)
#     print("AI 評分流程完成")
#     print(f"抽樣檔案：{paths['sample_csv']}")
#     print(f"AI 評分檔案：{paths['ai_scored_csv']}")
#     print(f"分數統計 CSV：{paths['score_distribution_csv']}")
#     print(f"二元判定統計 CSV：{paths['binary_distribution_csv']}")
#     print("=" * 70)


# # ============================================================
# # 12. 主控流程
# # ============================================================


# def main():
#     if MODE == "ai":
#         run_ai_pipeline()

#     elif MODE == "sample":
#         run_dir = create_run_dir(RESULT_ROOT)
#         paths = get_output_paths(run_dir)

#         sample_from_main_testset(
#             input_csv=MAIN_TESTSET_CSV,
#             output_csv=paths["sample_csv"],
#             sample_size=SAMPLE_SIZE,
#             random_seed=RANDOM_SEED,
#         )

#         save_run_info(paths, run_dir)

#     elif MODE == "ai_score":
#         run_dir = create_run_dir(RESULT_ROOT)
#         paths = get_output_paths(run_dir)

#         scored_df = run_ai_scoring(
#             input_csv=MANUAL_SAMPLE_CSV,
#             output_csv=paths["ai_scored_csv"],
#             model=MODEL_NAME,
#             sleep_sec=SLEEP_SEC,
#         )

#         print_score_distribution(
#             df=scored_df,
#             score_column=SCORE_COLUMN,
#             score_output_csv=paths["score_distribution_csv"],
#             binary_output_csv=paths["binary_distribution_csv"],
#         )

#         save_run_info(paths, run_dir)

#     elif MODE == "count":
#         run_dir = create_run_dir(RESULT_ROOT)
#         paths = get_output_paths(run_dir)

#         count_scores(
#             input_csv=MANUAL_AI_SCORED_CSV,
#             score_column=SCORE_COLUMN,
#             score_output_csv=paths["score_distribution_csv"],
#             binary_output_csv=paths["binary_distribution_csv"],
#         )

#         save_run_info(paths, run_dir)

#     elif MODE == "compare":
#         run_dir = create_run_dir(RESULT_ROOT)
#         paths = get_output_paths(run_dir)

#         compare_ai_human(
#             input_csv=MANUAL_HUMAN_AI_SCORED_CSV,
#             output_csv=paths["compare_csv"],
#             metrics_output_csv=paths["compare_metrics_csv"],
#         )

#         save_run_info(paths, run_dir)

#     else:
#         raise ValueError(f"未知 MODE：{MODE}")


# if __name__ == "__main__":
#     main()

In [12]:
import os
import json
import time
from pathlib import Path
from datetime import datetime

import pandas as pd
from openai import OpenAI

# ============================================================
# 0. 使用者設定區：主要改這裡即可
# ============================================================

MODE = "compare"
# 可選：
# "ai"       自動執行：抽樣 + AI 評分 + 統計分布，並建立新的 v1/v2/v3
# "compare" 讀取指定日期與版本資料夾中的人工評分檔，並把結果寫回同一個版本資料夾
# "count"   只統計指定版本資料夾內 ai_scored_10.csv 的分數分布

MAIN_TESTSET_CSV = "ragas_eval_with_response_2026-05-22.csv"

RESULT_ROOT = "rag_ai_manual_eval_results"

SAMPLE_SIZE = 10
RANDOM_SEED = 42

MODEL_NAME = "gpt-4o-mini"
SLEEP_SEC = 0

SCORE_COLUMN = "ai_score"

# compare / count 用：指定要操作哪一個既有版本資料夾
# 例如你的資料夾是 rag_ai_manual_eval_results/20260522/v2
TARGET_DATE = "20260522"
TARGET_VERSION = "v1"

# compare 用：
# 如果你直接在該版本資料夾的 ai_scored_10.csv 新增 human_score 欄位並存檔，
# 就維持 None。
# 如果你另存成 human_ai_scored_10.csv，也可以填檔名。
HUMAN_SCORED_FILENAME = None
# HUMAN_SCORED_FILENAME = "human_ai_scored_10.csv"


# ============================================================
# 1. 評分 Prompt
# ============================================================

SYSTEM_PROMPT = """
你是一位協助碩士論文進行 RAG 系統評估的學術評分者。
請根據「使用者問題 user_input」、「RAG 系統回答 response」與「檢索內容 retrieved_contexts」評估回答品質。

請主要判斷：
1. RAG 系統回答是否有回應使用者問題。
2. RAG 系統回答的主要內容是否能由 retrieved_contexts 支持。
3. RAG 系統回答是否包含 retrieved_contexts 無法支持的幻覺、錯誤或過度推論。

評分標準如下：
1 分：回答與使用者問題無關，且無法由 retrieved_contexts 支持。
2 分：回答僅少部分回應問題，或大多數內容缺乏 retrieved_contexts 支持。
3 分：回答大致回應問題，但內容不完整，或部分敘述無法由 retrieved_contexts 明確支持。
4 分：回答能回應問題，且大部分內容可由 retrieved_contexts 支持，僅有輕微不完整或表述不精確。
5 分：回答完整回應問題，且主要內容皆可由 retrieved_contexts 清楚支持，無明顯幻覺或錯誤。

請只根據提供的 user_input、response 與 retrieved_contexts 評分，不要自行補充外部知識。
請輸出 JSON。
""".strip()


# ============================================================
# 2. 資料夾與路徑管理
# ============================================================


def create_new_run_dir(result_root: str = RESULT_ROOT) -> Path:
    """
    MODE='ai' 使用：
    建立新的版本資料夾，例如：
    rag_ai_manual_eval_results/20260522/v1
    rag_ai_manual_eval_results/20260522/v2
    """
    today = datetime.now().strftime("%Y%m%d")
    date_dir = Path(result_root) / today
    date_dir.mkdir(parents=True, exist_ok=True)

    existing_versions = [
        p
        for p in date_dir.iterdir()
        if p.is_dir() and p.name.startswith("v") and p.name[1:].isdigit()
    ]

    if not existing_versions:
        next_version = 1
    else:
        version_numbers = [int(p.name[1:]) for p in existing_versions]
        next_version = max(version_numbers) + 1

    run_dir = date_dir / f"v{next_version}"
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir


def get_existing_run_dir(
    result_root: str, target_date: str, target_version: str
) -> Path:
    """
    MODE='compare' 或 MODE='count' 使用：
    取得既有版本資料夾，不新增 v。
    """
    run_dir = Path(result_root) / target_date / target_version

    if not run_dir.exists():
        raise FileNotFoundError(f"找不到指定資料夾：{run_dir}")

    if not run_dir.is_dir():
        raise NotADirectoryError(f"指定路徑不是資料夾：{run_dir}")

    return run_dir


def get_output_paths(run_dir: Path) -> dict:
    return {
        "sample_csv": run_dir / "rag_sample_10.csv",
        "ai_scored_csv": run_dir / "ai_scored_10.csv",
        "score_distribution_csv": run_dir / "score_distribution.csv",
        "binary_distribution_csv": run_dir / "binary_distribution.csv",
        "run_info_json": run_dir / "run_info.json",
        "prompt_txt": run_dir / "prompt_used.txt",
        "compare_csv": run_dir / "compare_result_10.csv",
        "compare_metrics_csv": run_dir / "compare_result_10_metrics.csv",
    }


# ============================================================
# 3. 分數轉二元判定
# ============================================================


def score_to_binary(score):
    try:
        score = int(score)
    except Exception:
        return None

    if score in [1, 2, 3]:
        return "未達可接受回答"
    if score in [4, 5]:
        return "達可接受回答"
    return None


# ============================================================
# 4. 從主測試集隨機抽樣
# ============================================================


def sample_from_main_testset(input_csv, output_csv, sample_size=10, random_seed=42):
    df = pd.read_csv(input_csv, encoding="utf-8-sig")

    required_cols = ["user_input", "response", "retrieved_contexts"]
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise ValueError(f"主測試集缺少必要欄位：{missing_cols}")

    if "status" in df.columns:
        success_values = ["success", "completed", "ok", "SUCCESS", "Completed", "OK"]
        success_df = df[df["status"].isin(success_values)].copy()

        if len(success_df) >= sample_size:
            df_for_sample = success_df
        else:
            print("提醒：status 成功資料不足，將改從全部資料抽樣。")
            df_for_sample = df
    else:
        df_for_sample = df

    if len(df_for_sample) < sample_size:
        raise ValueError(
            f"可抽樣資料只有 {len(df_for_sample)} 筆，小於 sample_size={sample_size}"
        )

    sample_df = df_for_sample.sample(n=sample_size, random_state=random_seed).copy()

    sample_df.insert(0, "sample_id", [f"Q{i + 1}" for i in range(len(sample_df))])
    sample_df.insert(1, "original_index", sample_df.index)

    front_cols = [
        "sample_id",
        "original_index",
        "user_input",
        "response",
        "retrieved_contexts",
    ]

    if "retrieved_metadata" in sample_df.columns:
        front_cols.append("retrieved_metadata")

    if "status" in sample_df.columns:
        front_cols.append("status")

    other_cols = [col for col in sample_df.columns if col not in front_cols]
    sample_df = sample_df[front_cols + other_cols]

    sample_df.to_csv(output_csv, index=False, encoding="utf-8-sig")

    print(f"已從 {input_csv} 隨機抽出 {sample_size} 題")
    print(f"抽樣結果已儲存至：{output_csv}")
    print("抽樣 sample_id：")
    print(sample_df["sample_id"].tolist())

    return sample_df


# ============================================================
# 5. 單題 AI 評分
# ============================================================


def get_openai_client():
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise EnvironmentError(
            "找不到 OPENAI_API_KEY。請先設定環境變數 OPENAI_API_KEY。"
        )
    return OpenAI(api_key=api_key)


def evaluate_one_row(row, model="gpt-4o-mini"):
    client = get_openai_client()

    user_input = str(row.get("user_input", ""))
    response = str(row.get("response", ""))
    retrieved_contexts = str(row.get("retrieved_contexts", ""))

    user_prompt = f"""
【使用者問題 user_input】
{user_input}

【RAG 系統回答 response】
{response}

【檢索內容 retrieved_contexts】
{retrieved_contexts}

請根據「是否回應使用者問題」與「是否受到 retrieved_contexts 支持」給出 1 至 5 分評分。
""".strip()

    response_format = {
        "type": "json_schema",
        "json_schema": {
            "name": "rag_answer_validity_score",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "score": {
                        "type": "integer",
                        "description": "回答支持性與問題回應性評分，範圍為 1 到 5 分",
                        "minimum": 1,
                        "maximum": 5,
                    },
                    "reason": {
                        "type": "string",
                        "description": "簡短說明評分理由",
                    },
                },
                "required": ["score", "reason"],
                "additionalProperties": False,
            },
        },
    }

    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        response_format=response_format,
        temperature=0,
    )

    content = completion.choices[0].message.content
    result = json.loads(content)

    return result["score"], result["reason"]


# ============================================================
# 6. 批次 AI 評分
# ============================================================


def run_ai_scoring(input_csv, output_csv, model="gpt-4o-mini", sleep_sec=0):
    df = pd.read_csv(input_csv, encoding="utf-8-sig")

    required_cols = ["user_input", "response", "retrieved_contexts"]
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise ValueError(f"缺少必要欄位：{missing_cols}")

    if "sample_id" not in df.columns:
        df.insert(0, "sample_id", [f"Q{i + 1}" for i in range(len(df))])

    ai_scores = []
    ai_reasons = []

    for idx, row in df.iterrows():
        current_id = row.get("sample_id", f"Q{idx + 1}")
        print(f"正在評分第 {idx + 1}/{len(df)} 題：{current_id}")

        try:
            score, reason = evaluate_one_row(row, model=model)
        except Exception as e:
            score = None
            reason = f"評分失敗：{e}"

        ai_scores.append(score)
        ai_reasons.append(reason)

        time.sleep(sleep_sec)

    df["ai_score"] = ai_scores
    df["ai_reason"] = ai_reasons
    df["ai_binary"] = df["ai_score"].apply(score_to_binary)

    df.to_csv(output_csv, index=False, encoding="utf-8-sig")

    print(f"\nAI 評分完成，已儲存至：{output_csv}")
    return df


# ============================================================
# 7. 分數分布統計
# ============================================================


def build_score_distribution_tables(df, score_column):
    numeric_scores = pd.to_numeric(df[score_column], errors="coerce")

    score_df = (
        numeric_scores.value_counts()
        .reindex([1, 2, 3, 4, 5], fill_value=0)
        .rename_axis("score")
        .reset_index(name="count")
    )

    binary_column = score_column.replace("_score", "_binary")

    if binary_column not in df.columns:
        df[binary_column] = df[score_column].apply(score_to_binary)

    binary_df = (
        df[binary_column]
        .value_counts()
        .reindex(["未達可接受回答", "達可接受回答"], fill_value=0)
        .rename_axis("binary_label")
        .reset_index(name="count")
    )

    return score_df, binary_df


def print_score_distribution(
    df, score_column, score_output_csv=None, binary_output_csv=None
):
    score_df, binary_df = build_score_distribution_tables(df, score_column)

    print(f"\n{score_column} 分數分布：")
    print(score_df)

    print(f"\n{score_column.replace('_score', '_binary')} 二元判定分布：")
    print(binary_df)

    if score_output_csv:
        score_df.to_csv(score_output_csv, index=False, encoding="utf-8-sig")
        print(f"\n分數分布 CSV 已輸出：{score_output_csv}")

    if binary_output_csv:
        binary_df.to_csv(binary_output_csv, index=False, encoding="utf-8-sig")
        print(f"二元判定分布 CSV 已輸出：{binary_output_csv}")


# ============================================================
# 8. AI 與人工比較
# ============================================================


def compare_ai_human(input_csv, output_csv=None, metrics_output_csv=None):
    df = pd.read_csv(input_csv, encoding="utf-8-sig")

    required_cols = ["ai_score", "human_score"]
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise ValueError(
            f"缺少必要欄位：{missing_cols}。請先在 AI 評分檔中新增 human_score 欄位。"
        )

    df["ai_binary"] = df["ai_score"].apply(score_to_binary)
    df["human_binary"] = df["human_score"].apply(score_to_binary)

    valid_df = df.dropna(subset=["ai_binary", "human_binary"]).copy()

    tp = len(
        valid_df[
            (valid_df["human_binary"] == "達可接受回答")
            & (valid_df["ai_binary"] == "達可接受回答")
        ]
    )

    fp = len(
        valid_df[
            (valid_df["human_binary"] == "未達可接受回答")
            & (valid_df["ai_binary"] == "達可接受回答")
        ]
    )

    fn = len(
        valid_df[
            (valid_df["human_binary"] == "達可接受回答")
            & (valid_df["ai_binary"] == "未達可接受回答")
        ]
    )

    tn = len(
        valid_df[
            (valid_df["human_binary"] == "未達可接受回答")
            & (valid_df["ai_binary"] == "未達可接受回答")
        ]
    )

    total = tp + fp + fn + tn

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (
        2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    )
    accuracy = (tp + tn) / total if total > 0 else 0

    same_score_count = len(valid_df[valid_df["ai_score"] == valid_df["human_score"]])
    same_binary_count = len(valid_df[valid_df["ai_binary"] == valid_df["human_binary"]])

    exact_score_agreement = same_score_count / len(valid_df) if len(valid_df) > 0 else 0
    binary_agreement = same_binary_count / len(valid_df) if len(valid_df) > 0 else 0

    print("\n混淆矩陣：")
    print("以人工評分為標準答案，AI 評分為預測結果")
    print("正類：達可接受回答")
    print()
    print("                                  AI 判定")
    print("人工判定                    達可接受回答       未達可接受回答")
    print(f"達可接受回答       TP        {tp:<12} FN        {fn:<12}")
    print(f"未達可接受回答     FP        {fp:<12} TN        {tn:<12}")

    print("\n評估指標：")
    print(f"Precision（精確率）: {precision:.4f}")
    print(f"Recall（召回率）   : {recall:.4f}")
    print(f"F1 分數             : {f1:.4f}")
    print(f"Accuracy（正確率） : {accuracy:.4f}")

    print("\n一致性補充：")
    print(
        f"完全同分題數：{same_score_count}/{len(valid_df)}，比例：{exact_score_agreement:.4f}"
    )
    print(
        f"二元判定一致題數：{same_binary_count}/{len(valid_df)}，比例：{binary_agreement:.4f}"
    )

    metrics = pd.DataFrame(
        [
            {
                "TP": tp,
                "FP": fp,
                "FN": fn,
                "TN": tn,
                "Precision": precision,
                "Recall": recall,
                "F1": f1,
                "Accuracy": accuracy,
                "ExactScoreAgreement": exact_score_agreement,
                "BinaryAgreement": binary_agreement,
                "ValidN": len(valid_df),
            }
        ]
    )

    if output_csv:
        valid_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
        print(f"\n已輸出比較結果：{output_csv}")

    if metrics_output_csv:
        metrics.to_csv(metrics_output_csv, index=False, encoding="utf-8-sig")
        print(f"已輸出指標結果：{metrics_output_csv}")

    return metrics


# ============================================================
# 9. 儲存執行資訊與 Prompt
# ============================================================


def save_run_info(paths, run_dir, extra_info=None):
    run_info = {
        "mode": MODE,
        "main_testset_csv": MAIN_TESTSET_CSV,
        "result_root": RESULT_ROOT,
        "sample_size": SAMPLE_SIZE,
        "random_seed": RANDOM_SEED,
        "model_name": MODEL_NAME,
        "sleep_sec": SLEEP_SEC,
        "score_column": SCORE_COLUMN,
        "output_dir": str(run_dir),
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    if extra_info:
        run_info.update(extra_info)

    with open(paths["run_info_json"], "w", encoding="utf-8") as f:
        json.dump(run_info, f, ensure_ascii=False, indent=2)

    with open(paths["prompt_txt"], "w", encoding="utf-8") as f:
        f.write(SYSTEM_PROMPT)

    print(f"執行資訊已輸出：{paths['run_info_json']}")
    print(f"評分 Prompt 已輸出：{paths['prompt_txt']}")


# ============================================================
# 10. MODE = "ai"
# ============================================================


def run_ai_pipeline():
    run_dir = create_new_run_dir(RESULT_ROOT)
    paths = get_output_paths(run_dir)

    print("=" * 70)
    print("開始執行 AI 評分流程")
    print(f"輸出資料夾：{run_dir}")
    print("=" * 70)

    sample_from_main_testset(
        input_csv=MAIN_TESTSET_CSV,
        output_csv=paths["sample_csv"],
        sample_size=SAMPLE_SIZE,
        random_seed=RANDOM_SEED,
    )

    scored_df = run_ai_scoring(
        input_csv=paths["sample_csv"],
        output_csv=paths["ai_scored_csv"],
        model=MODEL_NAME,
        sleep_sec=SLEEP_SEC,
    )

    print_score_distribution(
        df=scored_df,
        score_column=SCORE_COLUMN,
        score_output_csv=paths["score_distribution_csv"],
        binary_output_csv=paths["binary_distribution_csv"],
    )

    save_run_info(
        paths,
        run_dir,
        extra_info={
            "sample_csv": str(paths["sample_csv"]),
            "ai_scored_csv": str(paths["ai_scored_csv"]),
            "score_distribution_csv": str(paths["score_distribution_csv"]),
            "binary_distribution_csv": str(paths["binary_distribution_csv"]),
        },
    )

    print("=" * 70)
    print("AI 評分流程完成")
    print(f"抽樣檔案：{paths['sample_csv']}")
    print(f"AI 評分檔案：{paths['ai_scored_csv']}")
    print(f"分數統計 CSV：{paths['score_distribution_csv']}")
    print(f"二元判定統計 CSV：{paths['binary_distribution_csv']}")
    print("=" * 70)


# ============================================================
# 11. MODE = "compare"
# ============================================================


def run_compare_pipeline():
    run_dir = get_existing_run_dir(
        result_root=RESULT_ROOT,
        target_date=TARGET_DATE,
        target_version=TARGET_VERSION,
    )
    paths = get_output_paths(run_dir)

    if HUMAN_SCORED_FILENAME:
        input_csv = run_dir / HUMAN_SCORED_FILENAME
    else:
        input_csv = paths["ai_scored_csv"]

    print("=" * 70)
    print("開始執行 AI 與人工評分比較")
    print(f"指定資料夾：{run_dir}")
    print(f"讀取檔案：{input_csv}")
    print("=" * 70)

    compare_ai_human(
        input_csv=input_csv,
        output_csv=paths["compare_csv"],
        metrics_output_csv=paths["compare_metrics_csv"],
    )

    save_run_info(
        paths,
        run_dir,
        extra_info={
            "compare_input_csv": str(input_csv),
            "compare_csv": str(paths["compare_csv"]),
            "compare_metrics_csv": str(paths["compare_metrics_csv"]),
        },
    )

    print("=" * 70)
    print("比較流程完成")
    print(f"比較結果：{paths['compare_csv']}")
    print(f"指標結果：{paths['compare_metrics_csv']}")
    print("=" * 70)


# ============================================================
# 12. MODE = "count"
# ============================================================


def run_count_pipeline():
    run_dir = get_existing_run_dir(
        result_root=RESULT_ROOT,
        target_date=TARGET_DATE,
        target_version=TARGET_VERSION,
    )
    paths = get_output_paths(run_dir)

    df = pd.read_csv(paths["ai_scored_csv"], encoding="utf-8-sig")

    print_score_distribution(
        df=df,
        score_column=SCORE_COLUMN,
        score_output_csv=paths["score_distribution_csv"],
        binary_output_csv=paths["binary_distribution_csv"],
    )

    save_run_info(
        paths,
        run_dir,
        extra_info={
            "count_input_csv": str(paths["ai_scored_csv"]),
            "score_distribution_csv": str(paths["score_distribution_csv"]),
            "binary_distribution_csv": str(paths["binary_distribution_csv"]),
        },
    )


# ============================================================
# 13. 主控流程
# ============================================================


def main():
    if MODE == "ai":
        run_ai_pipeline()

    elif MODE == "compare":
        run_compare_pipeline()

    elif MODE == "count":
        run_count_pipeline()

    else:
        raise ValueError(f"未知 MODE：{MODE}")


if __name__ == "__main__":
    main()

開始執行 AI 與人工評分比較
指定資料夾：rag_ai_manual_eval_results\20260522\v1
讀取檔案：rag_ai_manual_eval_results\20260522\v1\ai_scored_10.csv

混淆矩陣：
以人工評分為標準答案，AI 評分為預測結果
正類：達可接受回答

                                  AI 判定
人工判定                    達可接受回答       未達可接受回答
達可接受回答       TP        8            FN        0           
未達可接受回答     FP        1            TN        1           

評估指標：
Precision（精確率）: 0.8889
Recall（召回率）   : 1.0000
F1 分數             : 0.9412
Accuracy（正確率） : 0.9000

一致性補充：
完全同分題數：7/10，比例：0.7000
二元判定一致題數：9/10，比例：0.9000

已輸出比較結果：rag_ai_manual_eval_results\20260522\v1\compare_result_10.csv
已輸出指標結果：rag_ai_manual_eval_results\20260522\v1\compare_result_10_metrics.csv
執行資訊已輸出：rag_ai_manual_eval_results\20260522\v1\run_info.json
評分 Prompt 已輸出：rag_ai_manual_eval_results\20260522\v1\prompt_used.txt
比較流程完成
比較結果：rag_ai_manual_eval_results\20260522\v1\compare_result_10.csv
指標結果：rag_ai_manual_eval_results\20260522\v1\compare_result_10_metrics.csv
